# 05 — Final Load & Tableau Prep

This notebook prepares the cleaned Airbnb dataset for Tableau visualisation and generates any additional derived columns needed for the final report.

**Input:**  (32,585 rows × 36 cols)

**Outputs:**
-  — Tableau-optimised export with derived KPI columns
-  — Neighbourhood-level aggregates for choropleth
-  — Room-type breakdown for bar/pie charts

## 1. Setup & Load

In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/cleaned_airbnb.csv', low_memory=False)
df['host_is_superhost'] = df['host_is_superhost'].map({'t': 1, 'f': 0}).fillna(0)
df['instant_bookable'] = df['instant_bookable'].map({'t': 1, 'f': 0}).fillna(0)

# Strip $ from price to work with numeric values
df['price'] = df['price'].astype(str).str.replace('[$,]', '', regex=True).astype(float)

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Loaded: 42,904 rows × 36 columns


,review_scores_accuracy,host_acceptance_rate,availability_90,neighbourhood_cleansed,availability_60,bedrooms,review_scores_cleanliness,host_response_time,reviews_per_month,amenities,...,host_since,number_of_reviews,review_scores_location,price,review_scores_communication,number_of_reviews_ltm,review_scores_value,availability_30,beds,availability_365
0,3.44,100.0,46.0,THIRD WARD,16.0,2.0,3.56,Unknown,0.06,"[""Smoke alarm"", ""Kitchen"", ""Air conditioning"",...",...,Unknown,9.0,3.22,147.0,4.56,0.0,3.67,0.0,2.0,321.0
1,4.88,100.0,26.0,SIXTH WARD,0.0,1.0,4.87,Unknown,2.26,"[""Smoke alarm"", ""Kitchen"", ""Dishwasher"", ""Wifi...",...,Unknown,317.0,4.82,147.0,4.81,10.0,4.78,0.0,1.0,301.0
2,4.61,100.0,35.0,SECOND WARD,31.0,0.0,4.45,Unknown,2.96,"[""Smoke alarm"", ""Kitchen"", ""Wifi"", ""Shower gel...",...,Unknown,389.0,4.76,147.0,4.86,20.0,4.63,18.0,2.0,35.0


## 2. Derive Segment Columns

These columns are **segments** (labels/categories) used to group and filter our real KPIs in Tableau. They are NOT KPIs themselves.

| Column | Type | Purpose |
|--------|------|----------|
| `price_tier` | Segment | Group listings by nightly price band |
| `availability_tier` | Segment | Group listings by annual availability |
| `revenue_tier` | Segment | Group listings by estimated revenue band |
| `rating_category` | Segment | Group listings by guest rating quality |

The **real KPIs** (numeric measures for Tableau) are: `price`, `estimated_revenue_l365d`, `estimated_occupancy_l365d`, `revpar`, `investment_potential`

In [32]:
# ── Price tier ─────────────────────────────────────────────────────────────
df['price_tier'] = pd.cut(
    df['price'],
    bins=[0, 75, 150, 300, 600, float('inf')],
    labels=['Budget (<5)', 'Mid (5–150)', 'Mid-High (50–300)', 'High (00–600)', 'Luxury (>00)']
)

# ── Availability tier ──────────────────────────────────────────────────────
df['availability_tier'] = pd.cut(
    df['availability_365'],
    bins=[-1, 30, 90, 180, 270, 365],
    labels=['Very Low (0–30d)', 'Low (31–90d)', 'Medium (91–180d)', 'High (181–270d)', 'Very High (271–365d)']
)

# ── Revenue tier ───────────────────────────────────────────────────────────
df['revenue_tier'] = pd.cut(
    df['estimated_revenue_l365d'],
    bins=[-1, 0, 5000, 20000, 50000, float('inf')],
    labels=['No Revenue', 'Low (<k)', 'Mid (k–20k)', 'High (0k–50k)', 'Top Earner (>0k)']
)

# ── Rating category ────────────────────────────────────────────────────────
df['rating_category'] = pd.cut(
    df['review_scores_rating'],
    bins=[0, 4.0, 4.5, 4.75, 4.9, 5.01],
    labels=['Poor (<4.0)', 'Good (4.0–4.5)', 'Very Good (4.5–4.75)', 'Excellent (4.75–4.9)', 'Perfect (4.9–5.0)']
)

print('Segment columns added:', ['price_tier', 'availability_tier', 'revenue_tier', 'rating_category'])
df[['price', 'price_tier', 'estimated_revenue_l365d', 'revenue_tier', 'review_scores_rating', 'rating_category']].head()

KPI columns added: ['price_tier', 'availability_tier', 'revenue_tier', 'rating_category']


,price,price_tier,estimated_revenue_l365d,revenue_tier,review_scores_rating,rating_category
0,147.0,Mid (5–150),4620.0,Low (<k),3.56,Poor (<4.0)
1,147.0,Mid (5–150),4620.0,Low (<k),4.75,Very Good (4.5–4.75)
2,147.0,Mid (5–150),4620.0,Low (<k),4.51,Very Good (4.5–4.75)
3,147.0,Mid (5–150),4620.0,Low (<k),4.87,Excellent (4.75–4.9)
4,147.0,Mid (5–150),4620.0,Low (<k),4.73,Very Good (4.5–4.75)


## 3. Neighbourhood Summary (for Choropleth / Bar)

In [33]:
neighbourhood_summary = (
    df.groupby('neighbourhood_cleansed')
    .agg(
        listing_count=('price', 'count'),
        median_price=('price', 'median'),
        mean_price=('price', 'mean'),
        median_revenue=('estimated_revenue_l365d', 'median'),
        mean_revenue=('estimated_revenue_l365d', 'mean'),
        median_occupancy=('estimated_occupancy_l365d', 'median'),
        avg_rating=('review_scores_rating', 'mean'),
        pct_superhost=('host_is_superhost', 'mean'),
        pct_instant_bookable=('instant_bookable', 'mean'),
    )
    .reset_index()
    .sort_values('median_revenue', ascending=False)
)

neighbourhood_summary['pct_superhost'] = neighbourhood_summary['pct_superhost'].round(3)
neighbourhood_summary['pct_instant_bookable'] = neighbourhood_summary['pct_instant_bookable'].round(3)
neighbourhood_summary['avg_rating'] = neighbourhood_summary['avg_rating'].round(2)

print(f'Neighbourhood summary: {neighbourhood_summary.shape}')
neighbourhood_summary.head(10)

Neighbourhood summary: (194, 10)


,neighbourhood_cleansed,listing_count,median_price,mean_price,median_revenue,mean_revenue,median_occupancy,avg_rating,pct_superhost,pct_instant_bookable
19,Bang Sue,193,720.0,1448.243523,21060.0,65619.626943,36.0,4.80,0.373,0.212
142,Vadhana,1430,1473.0,2726.732168,20697.0,129290.312587,18.0,4.74,0.313,0.397
123,Schoonbroek-Rozemaai,1,69.0,69.000000,16560.0,16560.000000,240.0,4.92,1.000,0.000
74,Khlong Toei,1299,1205.0,1761.078522,15660.0,111103.909931,18.0,4.70,0.298,0.467
111,Polder,2,1016.0,1016.000000,13728.0,13728.000000,33.0,4.84,0.000,0.500
36,Dam,12,136.5,140.916667,13422.0,14923.833333,114.0,4.83,0.250,0.417
106,Parthum Wan,369,1538.0,2198.728997,12192.0,183200.219512,14.0,4.74,0.347,0.393
77,Kruininge - Bremweide,3,147.0,139.000000,11400.0,10590.000000,90.0,4.80,0.000,0.000
52,Ekeren Centrum,4,131.0,142.500000,9603.0,10105.500000,72.0,4.88,0.750,0.000
47,Donk,3,120.0,114.000000,9600.0,9872.000000,80.0,4.78,0.667,0.000


## 4. Room-Type Summary

In [34]:
room_type_summary = (
    df.groupby('room_type')
    .agg(
        count=('price', 'count'),
        median_price=('price', 'median'),
        mean_price=('price', 'mean'),
        median_revenue=('estimated_revenue_l365d', 'median'),
        avg_rating=('review_scores_rating', 'mean'),
        median_occupancy=('estimated_occupancy_l365d', 'median'),
    )
    .reset_index()
    .sort_values('count', ascending=False)
)

room_type_summary['pct_of_market'] = (room_type_summary['count'] / room_type_summary['count'].sum() * 100).round(1)

print('Room-type summary:')
room_type_summary

Room-type summary:


,room_type,count,median_price,mean_price,median_revenue,avg_rating,median_occupancy,pct_of_market
0,Entire home/apt,34261,147.0,602.811360,4620.0,4.798056,24.0,79.9
2,Private room,7955,147.0,835.996229,4620.0,4.745415,0.0,18.5
1,Hotel room,519,160.0,1318.379576,4620.0,4.701503,0.0,1.2
3,Shared room,167,200.0,273.886228,216.0,4.784072,0.0,0.4
4,Unknown,2,147.0,147.000000,4620.0,4.870000,18.0,0.0


## 4. Advanced Business Metrics

To help the business sector evaluate investments, we calculate RevPAR and a weighted Investment Potential Score.

In [35]:
# 6. Advanced Business KPIs (RevPAR & Potential Score)
# RevPAR = Revenue Per Available Room (Standard Hotel/Short-term rental metric)
df["revpar"] = (df["price"] * df["estimated_occupancy_l365d"]) / 365

# Normalized Potential Score (0-100) based on Revenue, Rating and Frequency
# We use a simple weighted approach for business investment ranking
df["investment_potential"] = (
    (df["review_scores_rating"] / 5 * 40) +          # 40% Weight on Quality
    (df["estimated_occupancy_l365d"] / 365 * 40) +  # 40% Weight on Demand
    (np.log1p(df["estimated_revenue_l365d"]) / 12 * 20) # 20% Weight on Revenue log
).round(2)

print("Advanced KPIs added: [revpar, investment_potential]")
df[["neighbourhood_cleansed", "price", "revpar", "investment_potential"]].head()

Advanced KPIs added: [revpar, investment_potential]


,neighbourhood_cleansed,price,revpar,investment_potential
0,THIRD WARD,147.0,0.000000,42.54
1,SIXTH WARD,147.0,24.164384,58.64
2,SECOND WARD,147.0,48.328767,63.29
3,THIRTEENTH WARD,147.0,0.000000,53.02
4,SIXTH WARD,147.0,2.416438,52.56


## 5. Export Final Files

In [36]:
# ── 5a. Full Tableau-ready dataset ─────────────────────────────────────────
tableau_cols = [
    # Geography
    'neighbourhood_cleansed', 'latitude', 'longitude',
    # Property details
    'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    # Pricing & Revenue (KPIs)
    'price', 'estimated_revenue_l365d', 'estimated_occupancy_l365d',
    # Availability
    'availability_365', 'minimum_nights', 'maximum_nights',
    # Review scores
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    # Demand signals
    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_ly',
    'reviews_per_month',
    # Host attributes
    'host_is_superhost', 'host_acceptance_rate', 'instant_bookable',
    # Segment Columns (for filtering/grouping in Tableau)
    'price_tier', 'availability_tier', 'revenue_tier', 'rating_category',
    # Advanced KPIs (numeric measures)
    'revpar', 'investment_potential',
]

tableau_df = df[[c for c in tableau_cols if c in df.columns]].copy()

# Convert segment categoricals to strings for Tableau compatibility
for col in ['price_tier', 'availability_tier', 'revenue_tier', 'rating_category']:
    if col in tableau_df.columns:
        tableau_df[col] = tableau_df[col].astype(str)

tableau_path = '../data/processed/tableau_ready.csv'
tableau_df.to_csv(tableau_path, index=False)
print(f'Tableau dataset saved: {tableau_df.shape[0]:,} rows × {tableau_df.shape[1]} cols → {tableau_path}')

# ── 5b. Neighbourhood summary ──────────────────────────────────────────────
nbhd_path = '../data/processed/neighbourhood_summary.csv'
neighbourhood_summary.to_csv(nbhd_path, index=False)
print(f'Neighbourhood summary saved → {nbhd_path}')

# ── 5c. Room-type summary ──────────────────────────────────────────────────
room_path = '../data/processed/room_type_summary.csv'
room_type_summary.to_csv(room_path, index=False)
print(f'Room-type summary saved → {room_path}')

Tableau dataset saved: 42,904 rows × 33 cols → ../data/processed/tableau_ready.csv
Neighbourhood summary saved → ../data/processed/neighbourhood_summary.csv
Room-type summary saved → ../data/processed/room_type_summary.csv


## Summary

| Output File | Rows | Description |
|-------------|------|-------------|
| `tableau_ready.csv` | 42,904 | Full listing-level data optimised for Tableau |
| `neighbourhood_summary.csv` | ~194 | Aggregated by neighbourhood — choropleth / bar |
| `room_type_summary.csv` | 4 | Aggregated by room type — pie / bar |

### ✅ Real KPIs (numeric measures — what you measure)
| KPI | Column | Description |
|-----|--------|-------------|
| Nightly Price | `price` | Price per night in local currency |
| Estimated Revenue | `estimated_revenue_l365d` | Annual revenue estimate |
| Occupancy Rate | `estimated_occupancy_l365d` | Booked nights per year |
| RevPAR | `revpar` | Revenue Per Available Room (price × occupancy / 365) |
| Investment Score | `investment_potential` | Weighted score: quality 40% + demand 40% + revenue 20% |

### 🏷️ Segment Columns (for filtering/grouping — NOT KPIs)
| Segment | Column | Values |
|---------|--------|--------|
| Price Band | `price_tier` | Budget / Mid / Mid-High / High / Luxury |
| Availability Band | `availability_tier` | Very Low → Very High |
| Revenue Band | `revenue_tier` | No Revenue → Top Earner |
| Rating Quality | `rating_category` | Poor → Perfect |